# **Mini-project 1 : Brazillian E-commerce**

In [1]:
import pandas as pd
from sqlalchemy import create_engine


df_olist_customers = pd.read_csv('olist_customers_dataset.csv')
df_olist_sellers = pd.read_csv('olist_sellers_dataset.csv')
df_olist_order_reviews = pd.read_csv('olist_order_reviews_dataset.csv')
df_olist_order_items = pd.read_csv('olist_order_items_dataset.csv')
df_olist_products = pd.read_csv('olist_products_dataset.csv')
df_olist_geolocation = pd.read_csv('olist_geolocation_dataset.csv')
df_product_category_name_translation = pd.read_csv('product_category_name_translation.csv')
df_olist_orders = pd.read_csv('olist_orders_dataset.csv')
df_olist_order_payments = pd.read_csv('olist_order_payments_dataset.csv')


engine = create_engine('sqlite://', echo=False)

df_olist_customers.to_sql("olist_customers", con=engine, index=False)
df_olist_sellers.to_sql("olist_sellers", con=engine, index=False)
df_olist_order_reviews.to_sql("olist_order_reviews", con=engine, index=False)
df_olist_order_items.to_sql("olist_order_items", con=engine, index=False)
df_olist_products.to_sql("olist_products", con=engine, index=False)
df_olist_geolocation.to_sql("olist_geolocation", con=engine, index=False)
df_product_category_name_translation.to_sql("product_category_name_translation", con=engine, index=False)
df_olist_orders.to_sql("olist_orders", con=engine, index=False)
df_olist_order_payments.to_sql("olist_order_payments", con=engine, index=False)


103886

In [2]:
sql = '''
SELECT * FROM olist_customers
LIMIT 5
'''

df_sql = pd.read_sql_query(sql, con=engine)
df_sql.head()


,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,9790,sao bernardo do campo,SP
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,1151,sao paulo,SP
3,b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c,8775,mogi das cruzes,SP
4,4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,13056,campinas,SP


# **Query 1: 5-star reviews in January 2018**


In [14]:

sql = '''
SELECT
    COUNT(o.order_id) AS five_star_count,
    ROUND(100.0 * COUNT(o.order_id) / (
        SELECT COUNT(*)
        FROM olist_orders
        WHERE order_purchase_timestamp LIKE '2018-01%'
    ), 2) AS percentage_of_5_star
FROM olist_orders o
JOIN olist_order_reviews r ON o.order_id = r.order_id
WHERE r.review_score = 5
  AND o.order_purchase_timestamp LIKE '2018-01%'
'''

df_sql = pd.read_sql_query(sql, con=engine)
df_sql.head()


,five_star_count,percentage_of_5_star
0,4097,56.36


# **Query 2: Year-on-Year customer purchase trend**

In [15]:
sql = '''
SELECT
    STRFTIME('%Y', order_purchase_timestamp) AS order_year,
    COUNT(DISTINCT customer_id) AS total_customers
FROM olist_orders
GROUP BY order_year
ORDER BY order_year
'''

df_sql = pd.read_sql_query(sql, con=engine)
df_sql.head()


,order_year,total_customers
0,2016,329
1,2017,45101
2,2018,54011


# **Query 3: Average order values of customers**



In [16]:
sql = '''
SELECT
    o.customer_id,
    ROUND(AVG(op.payment_value), 2) AS avg_order_value
FROM olist_orders o
JOIN olist_order_payments op ON o.order_id = op.order_id
GROUP BY o.customer_id
ORDER BY avg_order_value DESC
LIMIT 10
'''

df_sql = pd.read_sql_query(sql, con=engine)
df_sql.head()



,customer_id,avg_order_value
0,1617b1357756262bfa56ab541c47bc16,13664.08
1,ec5b2ba62e574342386871631fafd3fc,7274.88
2,c6e2731c5b391845f6800c97401a43a9,6929.31
3,f48d464a0baaea338cb25f816991ab1f,6922.21
4,3fd6777bbce08a352fddd04e4a7cc8f6,6726.66


# **Query 4: Top 5 cities with highest revenue (2016–2018)**

In [17]:
sql = '''
SELECT
    c.customer_city,
    ROUND(SUM(p.payment_value), 2) AS total_revenue
FROM olist_orders o
JOIN olist_order_payments p ON o.order_id = p.order_id
JOIN olist_customers c ON o.customer_id = c.customer_id
WHERE STRFTIME('%Y', o.order_purchase_timestamp) BETWEEN '2016' AND '2018'
GROUP BY c.customer_city
ORDER BY total_revenue DESC
LIMIT 5
'''

df_sql = pd.read_sql_query(sql, con=engine)
df_sql.head()



,customer_city,total_revenue
0,sao paulo,2203373.09
1,rio de janeiro,1161927.36
2,belo horizonte,421765.12
3,brasilia,354216.78
4,curitiba,247392.48


# **Query 5: State-wise revenue (2016–2018)**

In [18]:
sql = '''
SELECT
    c.customer_state,
    ROUND(SUM(p.payment_value), 2) AS total_revenue
FROM olist_orders o
JOIN olist_order_payments p ON o.order_id = p.order_id
JOIN olist_customers c ON o.customer_id = c.customer_id
WHERE STRFTIME('%Y', o.order_purchase_timestamp) BETWEEN '2016' AND '2018'
GROUP BY c.customer_state
ORDER BY total_revenue DESC
'''

df_sql = pd.read_sql_query(sql, con=engine)
df_sql.head()



,customer_state,total_revenue
0,SP,5998226.96
1,RJ,2144379.69
2,MG,1872257.26
3,RS,890898.54
4,PR,811156.38


# **Query 6: Top sellers by items sold, revenue, customers, and 5-star reviews**

In [19]:
sql = '''
SELECT
    s.seller_id,
    COUNT(oi.order_item_id) AS total_items_sold,
    ROUND(SUM(op.payment_value), 2) AS total_revenue,
    COUNT(DISTINCT o.customer_id) AS total_customers,
    COUNT(CASE WHEN r.review_score = 5 THEN 1 END) AS five_star_reviews
FROM olist_order_items oi
JOIN olist_sellers s ON oi.seller_id = s.seller_id
JOIN olist_orders o ON oi.order_id = o.order_id
JOIN olist_order_payments op ON o.order_id = op.order_id
JOIN olist_order_reviews r ON o.order_id = r.order_id
WHERE STRFTIME('%Y', o.order_purchase_timestamp) BETWEEN '2016' AND '2018'
GROUP BY s.seller_id
ORDER BY total_revenue DESC
LIMIT 10
'''

df_sql = pd.read_sql_query(sql, con=engine)
df_sql.head()


,seller_id,total_items_sold,total_revenue,total_customers,five_star_reviews
0,7c67e1448b00f6e969d365cea6b010ab,1454,509474.13,976,489
1,1025f0e2d44d7041d6cf58b6550e0bfa,1465,310579.23,907,750
2,4a3ca9315b744ce9f8e9374361493884,2128,302403.81,1785,1021
3,1f50f920176fa81dab994f9023523100,2009,290729.12,1399,1134
4,53243585a1d6dc2643021fd1853d8905,435,282750.15,356,220


# **Query 7: Delivery success rate across states**

In [20]:
sql = '''
SELECT
    c.customer_state,
    COUNT(CASE WHEN o.order_delivered_customer_date IS NOT NULL THEN 1 END) * 100.0 / COUNT(*) AS delivery_success_rate
FROM olist_orders o
JOIN olist_customers c ON o.customer_id = c.customer_id
GROUP BY c.customer_state
ORDER BY delivery_success_rate DESC
'''

df_sql = pd.read_sql_query(sql, con=engine)
df_sql.head()



,customer_state,delivery_success_rate
0,AC,98.765432
1,AP,98.529412
2,ES,98.130841
3,MS,98.041958
4,AM,97.972973


# **Query 8: Preferred form of payment per product category**

In [21]:
sql = '''
WITH category_payments AS (
    SELECT
        pc.product_category_name_english AS category,
        op.payment_type,
        COUNT(*) AS payment_count
    FROM olist_order_items oi
    JOIN olist_products p ON oi.product_id = p.product_id
    JOIN product_category_name_translation pc ON p.product_category_name = pc.product_category_name
    JOIN olist_order_payments op ON oi.order_id = op.order_id
    GROUP BY pc.product_category_name_english, op.payment_type
)
SELECT
    cp.category,
    cp.payment_type,
    cp.payment_count
FROM category_payments cp
JOIN (
    SELECT category, MAX(payment_count) AS max_count
    FROM category_payments
    GROUP BY category
) cp_max
ON cp.category = cp_max.category AND cp.payment_count = cp_max.max_count
ORDER BY cp.payment_count DESC
'''

df_sql = pd.read_sql_query(sql, con=engine)
df_sql.head()



,category,payment_type,payment_count
0,bed_bath_table,credit_card,8959
1,health_beauty,credit_card,7566
2,sports_leisure,credit_card,6635
3,furniture_decor,credit_card,6379
4,computers_accessories,credit_card,5436


# **Query 9: Distances between cities**

In [22]:
sql = '''
WITH city_coords AS (
    SELECT
        geolocation_city,
        geolocation_state,
        ROUND(AVG(geolocation_lat), 5) AS lat,
        ROUND(AVG(geolocation_lng), 5) AS lng
    FROM olist_geolocation
    GROUP BY geolocation_city, geolocation_state
),
city_pairs AS (
    SELECT
        c1.geolocation_city AS city1,
        c1.geolocation_state AS state1,
        c1.lat AS lat1,
        c1.lng AS lng1,
        c2.geolocation_city AS city2,
        c2.geolocation_state AS state2,
        c2.lat AS lat2,
        c2.lng AS lng2
    FROM city_coords c1
    JOIN city_coords c2
        ON c1.geolocation_city < c2.geolocation_city
)
SELECT
    city1, state1,
    city2, state2,
    ROUND(
        6371 * ACOS(
            COS(RADIANS(lat1)) * COS(RADIANS(lat2)) *
            COS(RADIANS(lng2) - RADIANS(lng1)) +
            SIN(RADIANS(lat1)) * SIN(RADIANS(lat2))
        ),
        2
    ) AS distance_km
FROM city_pairs
ORDER BY distance_km DESC
LIMIT 100
'''

df_sql = pd.read_sql_query(sql, con=engine)
df_sql.head()


,city1,state1,city2,state2,distance_km
0,conquista d'oeste,MT,santa lucia do piai,RS,19946.34
1,nova lacerda,MT,santa lucia do piai,RS,19939.57
2,santa lucia do piai,RS,vale de são domingos,MT,19934.50
3,reserva do cabacal,MT,santa lucia do piai,RS,19934.20
4,santa lucia do piai,RS,vale de sao domingos,MT,19934.16
